<a href="https://colab.research.google.com/github/everjava/sctech/blob/feat%2Fto_class/Projeto_SCTEC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import csv
import re
from datetime import datetime
import json

class OlistPipeline:



  def __init__(self):
    self.SEM_CATEGORIA = 'sem categoria'
    self.CLEAN_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)
    self.products: list = []
    self.orders: list = []
    # Dicionário de estatísticas usado no relatório final (Tarefa 5)
    self._stats: dict = {
      "total_produtos_lidos": 0,
      "total_pedidos_lidos": 0,
      "categorias_preenchidas": 0,       # "Sem Categoria" inserido
      "dimensoes_corrigidas": 0,          # Campos físicos preenchidos com média
      "registros_descartados": 0,         # Produtos sem nenhuma dimensão
      "datas_entrega_nulas": 0,           # order_delivered_customer_date vazio
      "cancelados_sem_entrega": 0,        # Hipótese de negócio confirmada
      "nao_cancelados_sem_entrega": 0,    # Hipótese de negócio refutada
      "indisponivel": 0,                  # Pedido indisponível
      "datas_aprovacao_convertidas": 0,   # Datas reformatadas para pt-BR
      }


  def read_csv(self, path: str) -> list:
    print("read_csv")
    try:
      result = []
      with open(path, mode='r', encoding='utf-8') as arquivo:
        leitor_csv = csv.DictReader(arquivo)
        return list(leitor_csv)
    except FileNotFoundError:
      print(f"Erro: O arquivo '{path}' não foi encontrado.")
    except Exception as e:
      print(f"Ocorreu um erro: {e}")

  def clean_product_category_name(self, name: str) -> str:
    # print("clean_product_category_name")
    try:
      name = name.strip().lower()
      return self.CLEAN_PATTERN.sub('', name)
    except Exception as e:
      print(f"Ocorreu um erro: {e}")
      return None

  def convert_data_br(self, data: str) -> str:
    # print("convert_data_br")
    try:
      data_str = datetime.strptime(data, "%Y-%m-%d %H:%M:%S")
      return data_str.strftime("%d/%m/%Y")
    except Exception as e:
      print(f"Ocorreu um erro: {e}")
      return None


  def calculate_products_average(self, linha):
    # print("calculate_products_average")
    try:
      total_weight = 0.0
      total_length = 0.0
      total_height = 0.0
      total_width = 0.0

      # Contadores individuais para médias precisas (caso falte algum dado no CSV)
      count_w = count_l = count_h = count_wd = 0

      # for linha in leitor_csv:
        # 1. Processa Peso
      if linha.get('product_weight_g'):
        try:
          total_weight += float(linha['product_weight_g'])
          count_w += 1
        except ValueError:
          pass  # Ignora se não for um número válido

        # 2. Processa Comprimento
        if linha.get('product_length_cm'):
          try:
            total_length += float(linha['product_length_cm'])
            count_l += 1
          except ValueError:
            pass
        # 3. Processa Altura
        if linha.get('product_height_cm'):
          try:
            total_height += float(linha['product_height_cm'])
            count_h += 1
          except ValueError:
            pass
         # 4. Processa Largura
        if linha.get('product_width_cm'):
          try:
            total_width += float(linha['product_width_cm'])
            count_wd += 1
          except ValueError:
            pass

      # Calcula as médias (usa ternário para evitar ZeroDivisionError se o CSV estiver vazio)
      media_weight = total_weight / count_w if count_w > 0 else 0
      media_length = total_length / count_l if count_l > 0 else 0
      media_height = total_height / count_h if count_h > 0 else 0
      media_width = total_width / count_wd if count_wd > 0 else 0

      return media_weight, media_length, media_height, media_width
    except Exception as e:
      print(f"Ocorreu um erro: {e}")
      return None, None, None, None

  def execute_products(self, caminho_csv: str) -> list:

    try:
      self.products = self.read_csv(caminho_csv)
      for produto in self.products:

        if not produto['product_category_name']:
          # ----------------------------------------------------------------
          # TAREFA 1-A: Tratar categoria ausente
          # ----------------------------------------------------------------
          produto['product_category_name'] = self.SEM_CATEGORIA
        else:
          # ----------------------------------------------------------------
          # TAREFA 2: Tratar categoria ausente
          # ----------------------------------------------------------------
          produto['product_category_name'] = self.clean_product_category_name(produto['product_category_name'])

        # ----------------------------------------------------------------
        # TAREFA 1-B: Avaliar dimensões físicas
        # ----------------------------------------------------------------
        avg_weight_g, avg_length_cm, avg_height_cm, avg_width_cm = self.calculate_products_average(produto)
        if not produto['product_weight_g']:
          produto['product_weight_g'] = avg_weight_g

        if not produto['product_length_cm']:
          produto['product_length_cm'] = avg_length_cm

        if not produto['product_height_cm']:
          produto['product_height_cm'] = avg_height_cm

        if not produto['product_width_cm']:
          produto['product_width_cm'] = avg_width_cm


      return self.products
    except Exception as e:
      print(f"Ocorreu um erro: {e}")
      return None

  def execute_orders(self, caminho_csv: str) -> list:
    """
    "order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"
    e481f51cbdc54678b7cc49136f2d6af7,"9ef432eb6251297304e76186b10a928d",delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00

    approved  -----ok
    canceled ------ok
    invoiced ------ok
    processing-----ok
    shipped -------ok
    unavailable
    created
    delivered
    """
    try:
      self.orders = self.read_csv(caminho_csv)

      # ----------------------------------------------------------------
      # TAREFA 3: Hipótese — datas de entrega nulas ↔ pedidos cancelados?
      # ----------------------------------------------------------------
      for order in self.orders:
        order['order_purchase_timestamp'] = self.convert_data_br(order['order_purchase_timestamp'])
        order_delivered_customer_date = order.get("order_delivered_customer_date", "").strip()
        status = order.get("order_status", "").strip().lower()

        if not order_delivered_customer_date:
          order["order_delivered_customer_date"] = "N/A"
          self._stats["datas_entrega_nulas"] += 1

          if status == "canceled":
            # Hipótese CONFIRMADA para este registro:
            # sem data de entrega porque o pedido foi cancelado
            self._stats["cancelados_sem_entrega"] += 1
            order["delivery_hypothesis"] = "confirmed"

          elif status in ("shipped", "processing", "invoiced", "approved"):
            # Pedido ainda em andamento — entrega futura esperada
            order["delivery_hypothesis"] = "in_transit"
            self._stats["nao_cancelados_sem_entrega"] += 1

          elif status == "unavailable":
            # Pedido indisponível — entrega incerta
            order["delivery_hypothesis"] = "unavailable"
            self._stats["indisponivel"] += 1

          else:
            # Status desconhecido ou outro caso não mapeado
            order["delivery_hypothesis"] = "undefined"
            self._stats["nao_cancelados_sem_entrega"] += 1

        else:
          # Data de entrega presente — hipótese não aplicável
          order["delivery_hypothesis"] = "delivered"

      # ----------------------------------------------------------------
      # TAREFA 4: Reformatação de data de aprovação para padrão BR
      # ----------------------------------------------------------------
      order_approved_at = order.get("order_approved_at", "").strip()
      order["order_approved_at_br"] = self.convert_data_br(order_approved_at)

      if order["order_approved_at_br"]:
        self._stats["datas_aprovacao_convertidas"] += 1

      return self.orders


    except Exception as e:
      #TODO ADD LOGGER
      print(f"Ocorreu um erro: {e}")
      return None





def main():
    pipeline = OlistPipeline()
    # products = pipeline.execute_products('sample_data/olist_products_dataset.csv')
    # print(products)
    # beautiful_json = json.dumps(products, indent=4)
    # print(beautiful_json)

    orders = pipeline.execute_orders('sample_data/olist_orders_dataset.csv')
    beautiful_json = json.dumps(orders, indent=4)
    print(beautiful_json)

if __name__ == '__main__':
  main()






Streaming output truncated to the last 5000 lines.
        "order_delivered_customer_date": "2017-06-09 12:53:00",
        "order_estimated_delivery_date": "2017-06-22 00:00:00",
        "delivery_hypothesis": "delivered"
    },
    {
        "order_id": "244eddf1bb79ed749213ea45709f87cc",
        "customer_id": "27cdcc7fde4b4a2c57192e35fe14fc6c",
        "order_status": "delivered",
        "order_purchase_timestamp": "11/08/2018",
        "order_approved_at": "2018-08-11 15:24:28",
        "order_delivered_carrier_date": "2018-08-13 12:20:00",
        "order_delivered_customer_date": "2018-08-21 19:54:44",
        "order_estimated_delivery_date": "2018-08-22 00:00:00",
        "delivery_hypothesis": "delivered"
    },
    {
        "order_id": "5534625f6d8c8c073046dd7b57310af3",
        "customer_id": "df7bfa162625fe00fdee8139c5e4cd31",
        "order_status": "delivered",
        "order_purchase_timestamp": "19/04/2017",
        "order_approved_at": "2017-04-19 18:42:21",
        "o

In [4]:
import pandas as pd
import csv

with open('sample_data/olist_orders_dataset.csv', mode='r', encoding='utf-8') as arquivo:
  leitor_csv = csv.DictReader(arquivo)
  df = pd.DataFrame(leitor_csv)
  print(df.groupby('order_status').count())


              order_id  customer_id  order_purchase_timestamp  \
order_status                                                    
approved             2            2                         2   
canceled           625          625                       625   
created              5            5                         5   
delivered        96478        96478                     96478   
invoiced           314          314                       314   
processing         301          301                       301   
shipped           1107         1107                      1107   
unavailable        609          609                       609   

              order_approved_at  order_delivered_carrier_date  \
order_status                                                    
approved                      2                             2   
canceled                    625                           625   
created                       5                             5   
delivered               

In [23]:
import csv
import re


# Substitua 'seu_arquivo.csv' pelo caminho do seu arquivo
caminho_arquivo = 'sample_data/olist_products_dataset.csv'
SEM_CATEGORIA = 'Sem Categoria'
# Compila o padrão Regex para melhor desempenho em loops
# O padrão [^\w\s] encontra tudo o que NÃO é letra, número ou espaço (incluindo pontuações e caracteres especiais)
# O caractere '_' costuma ser incluído no \w, caso queira removê-lo também, use: r"[^\w\s]|_"
CLEAN_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)


try:
    with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
        # O DictReader transforma cada linha em um dicionário,
        # usando o cabeçalho como chave. Fica muito mais fácil de ler!
        leitor_csv = csv.DictReader(arquivo)

        # Ler todos os dados em uma lista para poder iterar várias vezes
        data = list(leitor_csv)

        print("Lendo os dados do arquivo:")
        print("-" * 30)

        # 2. Calcular a Média do comprimento do nome da categoria
        total_product_weight_g = 0
        total_product_length_cm = 0
        total_product_height_cm = 0
        total_product_width_cm = 0
        for linha in data:
            if 'product_weight_g' in linha and linha['product_weight_g']:
                total_product_weight_g += len(linha['product_weight_g'])

            if ('product_length_cm' in linha and linha['product_length_cm']):
                total_product_length_cm += len(linha['product_length_cm'])

            if ('product_height_cm' in linha and linha['product_height_cm']):
                total_product_height_cm += len(linha['product_height_cm'])

            if ('product_width_cm' in linha and linha['product_width_cm']):
                total_product_width_cm += len(linha['product_width_cm'])
        # Calcular a média

        quantidade = len(data)
        media_product_weight_g = total_product_weight_g / quantidade
        media_product_length_cm = total_product_length_cm / quantidade
        media_product_height_cm = total_product_height_cm / quantidade
        media_product_width_cm = total_product_width_cm / quantidade

        print(f"Número total de linhas: {quantidade}")
        print(f"Média do comprimento do nome da categoria: {media_product_weight_g:.2f}")
        print("-" * 30)

        for linha in data:
            if not linha['product_category_name']:
                linha['product_category_name'] = SEM_CATEGORIA
            else:
                linha['product_category_name'] = linha['product_category_name'].strip().lower()
                linha['product_category_name'] = CLEAN_PATTERN.sub('', linha['product_category_name'])

            if not linha['product_weight_g']:
                linha['product_weight_g'] = media_product_weight_g

            if not linha['product_length_cm']:
                linha['product_length_cm'] = media_product_length_cm

            if not linha['product_height_cm']:
                linha['product_height_cm'] = media_product_height_cm

            if not linha['product_width_cm']:
                linha['product_width_cm'] = media_product_width_cm


except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Lendo os dados do arquivo:
------------------------------
Número total de linhas: 32951
Média do comprimento do nome da categoria: 3.45
------------------------------
